**Probando conexión a Postgres**

In [6]:
from config.database import obtener_conexion

with obtener_conexion() as conn:
    with conn.cursor() as cursor:
        cursor.execute("SELECT version()")

        version = cursor.fetchone()

print(version)

('PostgreSQL 17.5 on x86_64-windows, compiled by msvc-19.44.35209, 64-bit',)


In [3]:
from config.database import obtener_conexion

query = """

CREATE TABLE usuarios (
    id_usuario SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    telefono VARCHAR(30),
    email VARCHAR(255) NOT NULL UNIQUE,
    password_hash VARCHAR(255) NOT NULL,
    rol VARCHAR(20) NOT NULL DEFAULT 'CLIENTE',
    fecha_creacion TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT chk_rol_usuario
        CHECK (
            rol IN (
                'ADMIN',
                'CLIENTE'
            )
        )
);


CREATE TABLE proveedores (
    id_proveedor SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    telefono VARCHAR(30),
    email VARCHAR(255)
);


CREATE TABLE servicios (
    id_servicio SERIAL PRIMARY KEY,
    nombre_servicio VARCHAR(100) NOT NULL,
    duracion_minutos INTEGER NOT NULL,
    precio NUMERIC(10,2) NOT NULL
);


CREATE TABLE turnos (
    id_turno SERIAL PRIMARY KEY,
    id_usuario INTEGER NOT NULL,
    id_proveedor INTEGER NOT NULL,
    id_servicio INTEGER NOT NULL,
    fecha_turno TIMESTAMP NOT NULL,
    estado VARCHAR(20) NOT NULL DEFAULT 'PENDIENTE',
    notas TEXT,
    fecha_creacion TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT fk_turno_usuario
        FOREIGN KEY (id_usuario)
        REFERENCES usuarios(id_usuario),

    CONSTRAINT fk_turno_proveedor
        FOREIGN KEY (id_proveedor)
        REFERENCES proveedores(id_proveedor),

    CONSTRAINT fk_turno_servicio
        FOREIGN KEY (id_servicio)
        REFERENCES servicios(id_servicio),

    CONSTRAINT chk_estado_turno
        CHECK (
            estado IN (
                'PENDIENTE',
                'CONFIRMADO',
                'COMPLETADO',
                'CANCELADO',
                'NO_ASISTIO'
            )
        )
);


CREATE TABLE pagos (
    id_pago SERIAL PRIMARY KEY,
    id_turno INTEGER NOT NULL,
    monto NUMERIC(10,2) NOT NULL,
    metodo_pago VARCHAR(30),
    estado VARCHAR(20) NOT NULL DEFAULT 'PENDIENTE',
    fecha_pago TIMESTAMP,
    fecha_creacion TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT fk_pago_turno
        FOREIGN KEY (id_turno)
        REFERENCES turnos(id_turno),

    CONSTRAINT chk_estado_pago
        CHECK (
            estado IN (
                'PENDIENTE',
                'PAGADO',
                'REEMBOLSADO'
            )
        )
);

        """

with obtener_conexion() as conn:
    with conn.cursor() as cursor:
        cursor.execute(query)

Prueba clientes

In [3]:
from repositories.clientes import crear_cliente
from repositories.clientes import obtener_clientes

crear_cliente(
    "Juan Pérez",
    "2611234567",
    "juan@gmail.com"
)

clientes = obtener_clientes()

print(clientes)

[(1, 'Juan Pérez', '2611234567', 'juan@gmail.com')]


In [5]:
from auth import hashear_password

password = "holaComoEstas"

print(hashear_password(password))

$2b$12$Z7GstwzfFbFTtuCOWcG8zuxgT7y.HWpkWwOnGCVt6yf2Nw.sIwdVG


### **Registrar usuario**

Funciona

In [4]:
from auth import registrar_usuario

registrar_usuario(
    nombre="Fran",
    telefono="2611111111",
    email="fran@gmail.com",
    password="Hola123!",
)

print("Usuario registrado")

Usuario registrado


No funciona

In [2]:
from auth import registrar_usuario

registrar_usuario(
    nombre="Carlos",
    telefono="2611111111",
    email="carlitos22222@gmail.com",
    password="hola1234",
)

print("Usuario registrado")

ValueError: La contraseña debe tener al menos 8 caracteres, una mayúscula y un carácter especial.

In [7]:
from repositories.usuarios import crear_usuario
from repositories.usuarios import obtener_usuarios

usuarios = obtener_usuarios()

print(usuarios)

[(2, 'Carlos', '2611111111', 'carlos@gmail.com', '$2b$12$npuuNuN5ctSy8beKSaZacOGdlzjoTgxL3xXpINh/WGUTcHOmGOy4i', 'CLIENTE', datetime.datetime(2026, 6, 19, 11, 51, 51, 323729)), (4, 'Carlos', '2611111111', 'carlitos22@gmail.com', '$2b$12$216g8Fi5cRDY7JfwzYpTVOHBiRuIFn0mQ3h6Egu14cY2aGDh/yiBm', 'CLIENTE', datetime.datetime(2026, 6, 19, 11, 53, 54, 665475)), (1, 'Fran', '2611111111', 'fran@gmail.com', '$2b$12$0XLRM7y1Sj5WXpxKAPPYD.R62rwcbI1ZIkF9myRUYmN/qgzW/xVgK', 'CLIENTE', datetime.datetime(2026, 6, 19, 11, 47, 0, 292473)), (6, 'Francesco', '2611111111', 'francescocornachione@gmail.com', '$2b$12$ovG2r1yG1OHVLmCaNNP3cOUiTHT4mmP9IgnIS8tbzDXkzxdZ7aswy', 'CLIENTE', datetime.datetime(2026, 6, 19, 16, 43, 57, 571506)), (7, 'Francesco', '2604', 'hola@gmail.com', '$2b$12$6Kuh2GnSAT4D4xZgJMDhFOGd4cXAKnqOW7B6SjnZ0hez3PuE/O6Qe', 'CLIENTE', datetime.datetime(2026, 6, 19, 16, 47, 22, 103006)), (8, 'yaie', '233', 'yair', '$2b$12$cp6.RyEqPlr86IP08ygFeOWUEcK/MKNG0Pa13hkA.y08it379caH2', 'CLIENTE', dateti

In [11]:
from auth import registrar_usuario, autenticar_usuario


def menu_cliente(usuario):

    while True:

        print("\n=== Menú Cliente ===")
        print("1. Reservar turno")
        print("2. Ver mis turnos")
        print("3. Cancelar turno")
        print("4. Editar mi perfil")
        print("5. Ver mis pagos")
        print("6. Cerrar sesión")

        opcion = input("> ")

        if opcion == "1":
            reservar_turno(usuario)

        elif opcion == "2":
            ver_mis_turnos(usuario)

        elif opcion == "3":
            cancelar_turno(usuario)

        elif opcion == "4":
            editar_perfil(usuario)

        elif opcion == "5":
            ver_mis_pagos(usuario)

        elif opcion == "6":
            print("\nSesión cerrada.")
            break

        else:
            print("Opción inválida.")


def main():

    while True:

        print("\n=== Sistema de Turnos ===")
        print("1. Registrarse")
        print("2. Iniciar sesión")
        print("3. Salir")

        opcion = input("> ")

        if opcion == "1":

            print("\n--- Registro ---")

            nombre = input("Nombre: ")
            telefono = input("Teléfono: ")
            email = input("Email: ")
            password = input("Contraseña: ")

            try:

                registrar_usuario(
                    nombre=nombre,
                    telefono=telefono,
                    email=email,
                    password=password
                )

                print("\nUsuario registrado correctamente.")

            except Exception as e:

                print(f"\nError: {e}")

        elif opcion == "2":

            print("\n--- Iniciar Sesión ---")

            email = input("Email: ")
            password = input("Contraseña: ")

            usuario = autenticar_usuario(
                email,
                password
            )

            if usuario:

                print("\nLogin exitoso")
                print(f"Bienvenido {usuario['nombre']}")

                menu_cliente(usuario)

            else:

                print("\nCredenciales incorrectas.")

        elif opcion == "3":

            print("\nHasta luego.")
            break

        else:

            print("\nOpción inválida.")


if __name__ == "__main__":
    main()


=== Sistema de Turnos ===
1. Registrarse
2. Iniciar sesión
3. Salir

--- Registro ---

Usuario registrado correctamente.

=== Sistema de Turnos ===
1. Registrarse
2. Iniciar sesión
3. Salir

--- Iniciar Sesión ---

Login exitoso
Bienvenido Francesco Cornachione

=== Menú Cliente ===
1. Reservar turno
2. Ver mis turnos
3. Cancelar turno
4. Editar mi perfil
5. Ver mis pagos
6. Cerrar sesión


NameError: name 'reservar_turno' is not defined